# CrossMed 02 - Train models
Run the controlled ResNet factorial experiment: weights frozen/trainable × head simple/MLP.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from crossmed.data import UltrasoundDataset
from crossmed.experiments import load_experiment_matrix, run_experiment_matrix
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MATRIX = PROJECT_ROOT / 'configs/experiment_matrix.yaml'
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Verify one transformed sample

In [ ]:
dataset = UltrasoundDataset(PROJECT_ROOT / 'data/manifests/busbra_train.csv', image_size=224, train=True, data_root=PROJECT_ROOT / 'data')
sample = dataset[0]
print('Tensor:', sample['image'].shape, 'Label:', sample['label'].item(), 'ID:', sample['sample_id'])

## Available experiments

In [ ]:
experiments = load_experiment_matrix(MATRIX)
pd.DataFrame({'experiment': experiments.keys(), 'configuration': experiments.values()})

## Select one experimental stage
Run fully trainable models first. The four ResNet groups form a controlled 2 × 2 comparison.

In [ ]:
FINETUNED_SIMPLE = ['resnet18_finetuned', 'resnet50_finetuned', 'resnet101_finetuned']
FINETUNED_MLP = ['resnet18_finetuned_mlp', 'resnet50_finetuned_mlp', 'resnet101_finetuned_mlp']
FROZEN_SIMPLE = ['resnet18_frozen_simple', 'resnet50_frozen_simple', 'resnet101_frozen_simple']
FROZEN_MLP = ['resnet18_frozen_mlp', 'resnet50_frozen_mlp', 'resnet101_frozen_mlp']
VIT = ['vit_b16_frozen']

SELECTED = FINETUNED_SIMPLE
FRESH_TRAINING = False
CLEANUP_BETWEEN_MODELS = True

## Train selected group
Existing checkpoints are skipped when `FRESH_TRAINING=False`. Set it to `True` only when intentionally replacing those results.

In [ ]:
training_status = run_experiment_matrix(
    MATRIX, selected=SELECTED, stage='train',
    skip_completed=not FRESH_TRAINING,
    cleanup_between_runs=CLEANUP_BETWEEN_MODELS,
)
training_status

Each run saves `best.pt` and `training_history.json` in its own output directory. With cleanup enabled, the live model is released before the next model loads.